In [1]:
import os
import boto3
import kagglehub
from botocore.client import Config

# 1. Download locally first (Kagglehub requirement)
local_raw_dir = '/opt/jupyter/data/'
bucket_name = 'imdb-bucket'

download_path = kagglehub.dataset_download(
    'ashirwadsangwan/imdb-dataset', 
    output_dir=local_raw_dir, 
    force_download=True
)
#2. Configure Boto3 client for MinIO
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000', # Your MinIO endpoint
    aws_access_key_id='lakehouse_admin',
    aws_secret_access_key='lakehouse_password',
    config=Config(signature_version='s3v4')
)
# Find the exact local TSV filename downloaded
tsv_files = [f for f in os.listdir(download_path) if f.endswith('.tsv')]

if not tsv_files:
    raise FileNotFoundError("No TSV file found in the downloaded archive.")

for tsv_file in tsv_files:
    
    local_tsv_path = os.path.join(download_path, tsv_file)
    minio_object_key = f"raw/{tsv_file}"
    s3_client.upload_file(local_tsv_path, bucket_name, minio_object_key)
    print(f"Successfully uploaded raw file to MinIO: s3://{bucket_name}/{minio_object_key}")
    os.remove(local_tsv_path)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 1.70G/1.70G [03:35<00:00, 8.47MB/s]

Extracting files...


Successfully uploaded raw file to MinIO: s3://imdb-bucket/raw/title.ratings.tsv
Successfully uploaded raw file to MinIO: s3://imdb-bucket/raw/title.principals.tsv
Successfully uploaded raw file to MinIO: s3://imdb-bucket/raw/title.akas.tsv
Successfully uploaded raw file to MinIO: s3://imdb-bucket/raw/name.basics.tsv
Successfully uploaded raw file to MinIO: s3://imdb-bucket/raw/title.basics.tsv


In [2]:
import shutil
from pathlib import Path

# Define your target path
dir_path = Path("/opt/jupyter/data/")

# Delete the directory tree permanently
if dir_path.exists() and dir_path.is_dir():
    shutil.rmtree(dir_path)